# Student Performance Dataset

## #01. 준비

### 1. 라이브러리 참조 및 데이터 불러오기

In [17]:
import numpy as np
from helpers import *
import seaborn as sb
import pandas as pd
from pandas import DataFrame, read_excel

In [18]:
# encoding='utf-8-sig'를 지정합니다.
origin = pd.read_csv(r'C:\Users\reqdr\Documents\VS works\student_performance_dataset.csv', encoding='utf-8-sig')
origin

,student_id,age,gender,city_type,study_hours_per_day,deep_work_sessions,assignment_completion_rate,attendance_percentage,social_media_hours,doomscrolling_before_sleep,...,family_support,financial_stress,learning_style,career_goal,productivity_after_midnight,revision_efficiency,burnout_risk,consistency_score,final_exam_score,performance_category
0,0,21,Female,Semi-Urban,3.2,7,100,70,3.8,0,...,10,6,Audio,Engineering,9,1,10,3,32,Low
1,1,19,Female,Semi-Urban,3.9,2,46,70,2.6,1,...,2,6,Practical,Business,6,10,4,6,59,Medium
2,2,16,Female,Urban,4.3,7,54,57,4.3,1,...,1,9,Visual,Engineering,7,9,8,1,34,Low
3,3,19,Male,Semi-Urban,5.3,1,78,90,1.7,0,...,5,10,Reading,Medical,7,7,9,10,60,Medium
4,4,17,Female,Urban,4.1,3,100,81,2.6,0,...,5,6,Visual,Medical,4,10,7,9,77,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2995,22,Female,Rural,2.4,0,40,83,1.0,0,...,10,2,Practical,Arts,9,10,7,9,45,Low
2996,2996,19,Female,Urban,5.2,7,94,100,1.4,0,...,1,10,Visual,Medical,5,3,7,9,100,High
2997,2997,23,Female,Rural,5.9,2,41,86,5.4,1,...,7,10,Practical,Engineering,6,8,10,10,50,Medium
2998,2998,17,Male,Urban,3.9,1,92,96,2.8,1,...,3,10,Practical,Medical,4,7,1,4,76,High


## #02. 자료형 점검

### 1. 자료형 확인

In [19]:
origin.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   student_id                   3000 non-null   int64  
 1   age                          3000 non-null   int64  
 2   gender                       3000 non-null   str    
 3   city_type                    3000 non-null   str    
 4   study_hours_per_day          3000 non-null   float64
 5   deep_work_sessions           3000 non-null   int64  
 6   assignment_completion_rate   3000 non-null   int64  
 7   attendance_percentage        3000 non-null   int64  
 8   social_media_hours           2940 non-null   float64
 9   doomscrolling_before_sleep   3000 non-null   int64  
 10  notification_distractions    3000 non-null   int64  
 11  ai_tool_usage_hours          3000 non-null   float64
 12  gaming_hours                 3000 non-null   float64
 13  stress_level                 

In [20]:
nominal = ['gender', 'city_type', 'doomscrolling_before_sleep', 'mental_state', 'learning_style', 'career_goal', 'performance_category'] 

### 2. 자료형 변환

In [21]:
df1 = origin.copy()
df1 = df1.dropna().reset_index(drop=True)

df1[nominal] = df1[nominal].astype('category')
df1['student_id'] = df1['student_id'].astype(str)
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   student_id                   2823 non-null   str     
 1   age                          2823 non-null   int64   
 2   gender                       2823 non-null   category
 3   city_type                    2823 non-null   category
 4   study_hours_per_day          2823 non-null   float64 
 5   deep_work_sessions           2823 non-null   int64   
 6   assignment_completion_rate   2823 non-null   int64   
 7   attendance_percentage        2823 non-null   int64   
 8   social_media_hours           2823 non-null   float64 
 9   doomscrolling_before_sleep   2823 non-null   category
 10  notification_distractions    2823 non-null   int64   
 11  ai_tool_usage_hours          2823 non-null   float64 
 12  gaming_hours                 2823 non-null   float64 
 13  stress_level  

## #03. 형식, 표기, 단위 점검

### 1. 명목형 변수의 단위 확인

In [22]:
df1[nominal].value_counts()

gender  city_type   doomscrolling_before_sleep  mental_state  learning_style  career_goal  performance_category
Male    Rural       1                           Balanced      Visual          Medical      Medium                  7
        Semi-Urban  0                           Burnout       Audio           Arts         Low                     6
                                                Distracted    Practical       Research     Medium                  6
                    1                           Burnout       Audio           Business     Low                     6
        Urban       0                           Burnout       Reading         Arts         Low                     6
                                                                                                                  ..
        Rural       1                           Distracted    Reading         Business     High                    0
                                                                     

### 2. 연속형 변수의 단위 확인을 위한 변수명 추출

In [23]:
# 값을 검사할 수치형 변수의 이름 추출
fields = df1.select_dtypes(include="number").columns.to_list()
print(fields)

['age', 'study_hours_per_day', 'deep_work_sessions', 'assignment_completion_rate', 'attendance_percentage', 'social_media_hours', 'notification_distractions', 'ai_tool_usage_hours', 'gaming_hours', 'stress_level', 'motivation_level', 'focus_score', 'procrastination_index', 'sleep_hours', 'caffeine_intake', 'physical_activity_hours', 'internet_quality', 'family_support', 'financial_stress', 'productivity_after_midnight', 'revision_efficiency', 'burnout_risk', 'consistency_score', 'final_exam_score']


### 3. 연속형 변수의 단위 확인

In [24]:
minmax = []     # 최소값과 최대값을 저장할 리스트

for field in fields:
    min_value = df1[field].min()
    max_value = df1[field].max()
    minmax.append({"min": min_value, "max": max_value})

# 최소값과 최대값을 데이터 프레임으로 변환하여 출력
minmax_df = DataFrame(minmax, index=fields)
minmax_df

,min,max
age,15.0,23.0
study_hours_per_day,0.5,8.8
deep_work_sessions,0.0,7.0
assignment_completion_rate,40.0,100.0
attendance_percentage,50.0,100.0
social_media_hours,0.0,8.9
notification_distractions,5.0,119.0
ai_tool_usage_hours,0.0,5.0
gaming_hours,0.0,7.0
stress_level,1.0,10.0


## #04. 데이터 중복 점검

### 1. 행 단위 완전 중복 확인

In [25]:
dup = df1.duplicated(keep=False)
dup

0       False
1       False
2       False
3       False
4       False
        ...  
2818    False
2819    False
2820    False
2821    False
2822    False
Length: 2823, dtype: bool

In [26]:
# 중복 데이터 개수
dup.sum()

np.int64(0)

## #05. 결측치 점검

### 1. 결측치 확인

In [27]:
na_count = df1.isna().sum()
na_count

student_id                     0
age                            0
gender                         0
city_type                      0
study_hours_per_day            0
deep_work_sessions             0
assignment_completion_rate     0
attendance_percentage          0
social_media_hours             0
doomscrolling_before_sleep     0
notification_distractions      0
ai_tool_usage_hours            0
gaming_hours                   0
stress_level                   0
motivation_level               0
focus_score                    0
procrastination_index          0
mental_state                   0
sleep_hours                    0
caffeine_intake                0
physical_activity_hours        0
internet_quality               0
family_support                 0
financial_stress               0
learning_style                 0
career_goal                    0
productivity_after_midnight    0
revision_efficiency            0
burnout_risk                   0
consistency_score              0
final_exam

### 2. 결측치 비율 계산을 위한 데이터 크기 확인

In [28]:
# 데이터의 행과 열의 수 확인
rows, cols = df1.shape
print(f"rows: {rows}, cols: {cols}")

rows: 2823, cols: 32


### 3. 결측치 비율 확인

In [29]:
na_ratio = na_count / rows
na_ratio

student_id                     0.0
age                            0.0
gender                         0.0
city_type                      0.0
study_hours_per_day            0.0
deep_work_sessions             0.0
assignment_completion_rate     0.0
attendance_percentage          0.0
social_media_hours             0.0
doomscrolling_before_sleep     0.0
notification_distractions      0.0
ai_tool_usage_hours            0.0
gaming_hours                   0.0
stress_level                   0.0
motivation_level               0.0
focus_score                    0.0
procrastination_index          0.0
mental_state                   0.0
sleep_hours                    0.0
caffeine_intake                0.0
physical_activity_hours        0.0
internet_quality               0.0
family_support                 0.0
financial_stress               0.0
learning_style                 0.0
career_goal                    0.0
productivity_after_midnight    0.0
revision_efficiency            0.0
burnout_risk        

## #06. 품질 점검 결과 저장

### 1. 엑셀 파일로 저장하기

In [30]:
df1.to_excel("student_perfomance_qtcheck.xlsx", index=False)

## #07. 기술 통계량 확인 준비 작업

### 1. 데이터 불러오기

In [31]:
origin_qt = read_excel("student_perfomance_qtcheck.xlsx")

### 2. 타입변환

In [32]:
# 타입변환은 데이터를 불러올때마다 수행해야 한다
df2 = origin_qt.copy()
df2 = df2.dropna().reset_index(drop=True)

df2[nominal] = df2[nominal].astype('category')
df2['student_id'] = df2['student_id'].astype(str)

df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   student_id                   2823 non-null   str     
 1   age                          2823 non-null   int64   
 2   gender                       2823 non-null   category
 3   city_type                    2823 non-null   category
 4   study_hours_per_day          2823 non-null   float64 
 5   deep_work_sessions           2823 non-null   int64   
 6   assignment_completion_rate   2823 non-null   int64   
 7   attendance_percentage        2823 non-null   int64   
 8   social_media_hours           2823 non-null   float64 
 9   doomscrolling_before_sleep   2823 non-null   category
 10  notification_distractions    2823 non-null   int64   
 11  ai_tool_usage_hours          2823 non-null   float64 
 12  gaming_hours                 2823 non-null   float64 
 13  stress_level  

## #08. 기술 통계량 표 생성

### 1. 연속형 변수의 기술 통계량 표

In [33]:
# T는 데이터 프레임의 행과 열을 바꿔주는 역할
desc_df = df2.describe().T
desc_df

,count,mean,std,min,25%,50%,75%,max
age,2823.0,19.033652,2.561767,15.0,17.0,19.0,21.0,23.0
study_hours_per_day,2823.0,4.004711,1.497795,0.5,2.9,4.0,5.0,8.8
deep_work_sessions,2823.0,3.512221,2.294983,0.0,1.0,4.0,6.0,7.0
assignment_completion_rate,2823.0,70.033652,17.580813,40.0,55.0,70.0,85.0,100.0
attendance_percentage,2823.0,74.503365,14.462931,50.0,62.0,74.0,87.0,100.0
social_media_hours,2823.0,2.939603,1.510860,0.0,1.9,2.9,4.0,8.9
notification_distractions,2823.0,61.874601,33.112813,5.0,32.0,62.0,91.0,119.0
ai_tool_usage_hours,2823.0,1.547644,0.925989,0.0,0.9,1.5,2.2,5.0
gaming_hours,2823.0,2.048742,1.384542,0.0,0.9,2.0,3.0,7.0
stress_level,2823.0,5.561105,2.890342,1.0,3.0,6.0,8.0,10.0


### 2. 명목형 변수의 기술 통계량 표

In [34]:
cate_desc_df = df2.describe(include='category').T
cate_desc_df

,count,unique,top,freq
gender,2823,2,Female,1434
city_type,2823,3,Rural,960
doomscrolling_before_sleep,2823,2,0,1421
mental_state,2823,4,Distracted,741
learning_style,2823,4,Visual,744
career_goal,2823,5,Arts,595
performance_category,2823,3,Medium,1280


### 3. 명목형 변수의 기술 통계량

In [35]:
# 명목형 변수의 필드명 추출
cate_fields = df2.select_dtypes(include='category').columns

# 명목형 변수의 고유값과 빈도수 계산
for field in cate_fields:
    # 값의 종류별 개수
    vcount = df2[field].value_counts()
    # 전체 데이터 길이 대비 비율
    percent = vcount / df2.shape[0]

    # 결과 저장
    df = DataFrame({'count': vcount, 'percent': percent})
    # 결과 출력
    display(df)

,count,percent
gender,,
Female,1434,0.50797
Male,1389,0.49203


,count,percent
city_type,,
Rural,960,0.340064
Semi-Urban,950,0.336521
Urban,913,0.323415


,count,percent
doomscrolling_before_sleep,,
0,1421,0.503365
1,1402,0.496635


,count,percent
mental_state,,
Distracted,741,0.262487
Burnout,712,0.252214
Focused,692,0.245129
Balanced,678,0.240170


,count,percent
learning_style,,
Visual,744,0.263549
Practical,716,0.253631
Audio,685,0.242650
Reading,678,0.240170


,count,percent
career_goal,,
Arts,595,0.210769
Medical,593,0.210060
Engineering,559,0.198016
Business,551,0.195182
Research,525,0.185972


,count,percent
performance_category,,
Medium,1280,0.453418
Low,1120,0.396741
High,423,0.149841


## #09. 중심 수준 파악

### 1. 평균-중앙값의 상대 차이율 계산

In [36]:
# "평균-중앙값 상대 차이율 = |평균 - 중앙값| / 중앙값" 컬럼 추가
desc_df['rel_diff'] = abs(desc_df['mean'] - desc_df['50%']) / desc_df['50%']

# 상대 차이율 의미 컬럼 추가
conditions = [desc_df['rel_diff'] < 0.1, desc_df['rel_diff'] < 0.5]
choices = ['similar', 'diff']
desc_df['rdiff_flag'] = np.select(conditions, choices, default='large_diff')

# 필요한 컬럼만 선택하여 출력
desc_df[['mean', '50%', 'rel_diff', 'rdiff_flag']]

,mean,50%,rel_diff,rdiff_flag
age,19.033652,19.0,0.001771,similar
study_hours_per_day,4.004711,4.0,0.001178,similar
deep_work_sessions,3.512221,4.0,0.121945,diff
assignment_completion_rate,70.033652,70.0,0.000481,similar
attendance_percentage,74.503365,74.0,0.006802,similar
social_media_hours,2.939603,2.9,0.013656,similar
notification_distractions,61.874601,62.0,0.002023,similar
ai_tool_usage_hours,1.547644,1.5,0.031763,similar
gaming_hours,2.048742,2.0,0.024371,similar
stress_level,5.561105,6.0,0.073149,similar


## #10. 이상치 신호 감지

### 1. IQR, 이상치 경계값 계산

In [37]:
# iqr
desc_df['iqr'] = desc_df['75%'] - desc_df['25%']

# 상한 이상치 경계
desc_df['upper_bound'] = desc_df['75%'] + 1.5 * desc_df['iqr']
desc_df['lower_bound'] = desc_df['25%'] - 1.5 * desc_df['iqr']

# 필요한 컬럼만 선택하여 출력
desc_df[['iqr', 'upper_bound', 'lower_bound']]

,iqr,upper_bound,lower_bound
age,4.0,27.00,11.00
study_hours_per_day,2.1,8.15,-0.25
deep_work_sessions,5.0,13.50,-6.50
assignment_completion_rate,30.0,130.00,10.00
attendance_percentage,25.0,124.50,24.50
social_media_hours,2.1,7.15,-1.25
notification_distractions,59.0,179.50,-56.50
ai_tool_usage_hours,1.3,4.15,-1.05
gaming_hours,2.1,6.15,-2.25
stress_level,5.0,15.50,-4.50


### 2. 명목형 변수를 제외한 데이터 프레임

In [38]:
# 명목형 변수의 필드명 추출
cate_fields = df2.select_dtypes(include='category').columns

df3 = df2.drop(columns=cate_fields)
df3 = df3.drop(columns=['student_id'])
df3.head()

,age,study_hours_per_day,deep_work_sessions,assignment_completion_rate,attendance_percentage,social_media_hours,notification_distractions,ai_tool_usage_hours,gaming_hours,stress_level,...,caffeine_intake,physical_activity_hours,internet_quality,family_support,financial_stress,productivity_after_midnight,revision_efficiency,burnout_risk,consistency_score,final_exam_score
0,21,3.2,7,100,70,3.8,107,0.6,1.8,8,...,4,0.6,1,10,6,9,1,10,3,32
1,19,3.9,2,46,70,2.6,93,1.5,1.1,4,...,5,0.0,4,2,6,6,10,4,6,59
2,16,4.3,7,54,57,4.3,82,0.5,0.0,2,...,4,1.3,9,1,9,7,9,8,1,34
3,19,5.3,1,78,90,1.7,27,1.4,3.8,8,...,2,1.3,1,5,10,7,7,9,10,60
4,17,4.1,3,100,81,2.6,8,1.3,2.5,4,...,5,1.7,9,5,6,4,10,7,9,77


### 3. 상한 이상치 탐지

In [39]:
# 상한 이상치 수
desc_df['upper_outliers'] = ((df3 > desc_df['upper_bound'])).sum()

# 상한 이상치 수 비율
desc_df['upper_outliers_ratio'] = desc_df['upper_outliers'] / df3.shape[0]

# 필요한 컬럼만 선택하여 출력
desc_df[['upper_outliers', 'upper_outliers_ratio']]

,upper_outliers,upper_outliers_ratio
age,0,0.000000
study_hours_per_day,6,0.002125
deep_work_sessions,0,0.000000
assignment_completion_rate,0,0.000000
attendance_percentage,0,0.000000
social_media_hours,5,0.001771
notification_distractions,0,0.000000
ai_tool_usage_hours,19,0.006730
gaming_hours,6,0.002125
stress_level,0,0.000000


### 4. 하한 이상치 탐지

In [40]:
# 하한 이상치 수
desc_df['lower_outliers'] = ((df3 < desc_df['lower_bound'])).sum()

# 하한 이상치 수 비율
desc_df['lower_outliers_ratio'] = desc_df['lower_outliers'] / df3.shape[0]

# 필요한 컬럼만 선택하여 출력
desc_df[['lower_outliers', 'lower_outliers_ratio']]

,lower_outliers,lower_outliers_ratio
age,0,0.000000
study_hours_per_day,0,0.000000
deep_work_sessions,0,0.000000
assignment_completion_rate,0,0.000000
attendance_percentage,0,0.000000
social_media_hours,0,0.000000
notification_distractions,0,0.000000
ai_tool_usage_hours,0,0.000000
gaming_hours,0,0.000000
stress_level,0,0.000000


### 5. 전체 이상치 집계

In [41]:
# 통합 이상치 수
desc_df['outliers'] = desc_df['upper_outliers'] + desc_df['lower_outliers']

# 통합 이상치 수 비율
desc_df['outliers_ratio'] = desc_df['outliers'] / df3.shape[0]

# 이상치 탐지 결과 확인
desc_df[['upper_outliers', 'upper_outliers_ratio', 'lower_outliers', 'lower_outliers_ratio', 'outliers', 'outliers_ratio']]

,upper_outliers,upper_outliers_ratio,lower_outliers,lower_outliers_ratio,outliers,outliers_ratio
age,0,0.000000,0,0.000000,0,0.000000
study_hours_per_day,6,0.002125,0,0.000000,6,0.002125
deep_work_sessions,0,0.000000,0,0.000000,0,0.000000
assignment_completion_rate,0,0.000000,0,0.000000,0,0.000000
attendance_percentage,0,0.000000,0,0.000000,0,0.000000
social_media_hours,5,0.001771,0,0.000000,5,0.001771
notification_distractions,0,0.000000,0,0.000000,0,0.000000
ai_tool_usage_hours,19,0.006730,0,0.000000,19,0.006730
gaming_hours,6,0.002125,0,0.000000,6,0.002125
stress_level,0,0.000000,0,0.000000,0,0.000000


## #11. 비대칭 신호 확인

### 1. 왜도 점검

In [42]:
# 왜도 계산
desc_df['skew'] = df3.skew()

# 왜도를 통한 분포 형태 해석
conditions_skew = [(desc_df['skew'] < -0.5), (desc_df['skew'] > 0.5)]
choices_skew = ['left tail', 'right tail']
desc_df['skew_interpret'] = np.select(conditions_skew, choices_skew, default='symmetric')

# 필요한 컬럼만 선택하여 출력
desc_df[['skew', 'skew_interpret']]

,skew,skew_interpret
age,0.013693,symmetric
study_hours_per_day,0.017044,symmetric
deep_work_sessions,0.006885,symmetric
assignment_completion_rate,-0.013736,symmetric
attendance_percentage,0.043500,symmetric
social_media_hours,0.147510,symmetric
notification_distractions,-0.022074,symmetric
ai_tool_usage_hours,0.324083,symmetric
gaming_hours,0.413781,symmetric
stress_level,-0.024867,symmetric


### 2. 첨도 점검

In [43]:
# 첨도 계산
desc_df['kurt'] = df3.kurt()

# 첨도를 통한 분포 형태 해석
conditions_kurt = [(desc_df['kurt'] < 0), (desc_df['kurt'] > 0)]
choices_kurt = ['playkurtic', 'leptokurtic']
desc_df['kurt_interpret'] = np.select(conditions_kurt, choices_kurt, default='mesokurtic')

# 필요한 컬럼만 선택하여 출력
desc_df[['kurt', 'kurt_interpret']]

,kurt,kurt_interpret
age,-1.209952,playkurtic
study_hours_per_day,-0.237435,playkurtic
deep_work_sessions,-1.242440,playkurtic
assignment_completion_rate,-1.199306,playkurtic
attendance_percentage,-1.156480,playkurtic
social_media_hours,-0.342058,playkurtic
notification_distractions,-1.222585,playkurtic
ai_tool_usage_hours,-0.147533,playkurtic
gaming_hours,-0.352858,playkurtic
stress_level,-1.219665,playkurtic


### 3. 로그 변환 필요성 판단 함수 정의

In [44]:
def judge_log_transform(skew, kurt):
    if skew >= 1:
        return "log1p"
    elif skew > 0.5 and kurt > 0:
        return "log1p"
    elif skew <= -1:
        return "reverse_log1p"
    elif skew < -0.5 and kurt > 0:
        return "reverse_log1p"
    else:
        return "none"

### 4. 로그 변환 필요성 판정

In [45]:
desc_df['log_need'] = desc_df.apply(lambda row: judge_log_transform(row['skew'], row['kurt']), axis=1)

# 로그 변환 필요성 결과 확인
desc_df[['skew', 'kurt', 'log_need']]

,skew,kurt,log_need
age,0.013693,-1.209952,none
study_hours_per_day,0.017044,-0.237435,none
deep_work_sessions,0.006885,-1.242440,none
assignment_completion_rate,-0.013736,-1.199306,none
attendance_percentage,0.043500,-1.156480,none
social_media_hours,0.147510,-0.342058,none
notification_distractions,-0.022074,-1.222585,none
ai_tool_usage_hours,0.324083,-0.147533,none
gaming_hours,0.413781,-0.352858,none
stress_level,-0.024867,-1.219665,none


## #12. 최종 기술 통계량

### 1. 최종 기술 통계량 확인

In [46]:
# 행(Row) 생략 없이 최대 30개까지 출력
pd.set_option("display.max_rows", 30)

# 열(Column) 생략 없이 최대 30개까지 출력
pd.set_option("display.max_columns", 30)

# 열 너비가 길어지더라도 줄바꿈 없이 한 줄에 출력
pd.set_option("display.width", 1000)

desc_df.T

,age,study_hours_per_day,deep_work_sessions,assignment_completion_rate,attendance_percentage,social_media_hours,notification_distractions,ai_tool_usage_hours,gaming_hours,stress_level,motivation_level,focus_score,procrastination_index,sleep_hours,caffeine_intake,physical_activity_hours,internet_quality,family_support,financial_stress,productivity_after_midnight,revision_efficiency,burnout_risk,consistency_score,final_exam_score
count,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0,2823.0
mean,19.033652,4.004711,3.512221,70.033652,74.503365,2.939603,61.874601,1.547644,2.048742,5.561105,5.535246,5.500177,5.58413,6.5096,3.511513,1.055685,5.515409,5.483528,5.580942,5.554021,5.429685,5.527453,5.532412,54.820758
std,2.561767,1.497795,2.294983,17.580813,14.462931,1.51086,33.112813,0.925989,1.384542,2.890342,2.856244,2.863895,2.881905,1.198051,2.27576,0.72969,2.911102,2.855173,2.861389,2.856943,2.891359,2.838159,2.858882,18.723068
min,15.0,0.5,0.0,40.0,50.0,0.0,5.0,0.0,0.0,1.0,1.0,1.0,1.0,3.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
25%,17.0,2.9,1.0,55.0,62.0,1.9,32.0,0.9,0.9,3.0,3.0,3.0,3.0,5.7,2.0,0.5,3.0,3.0,3.0,3.0,3.0,3.0,3.0,42.0
50%,19.0,4.0,4.0,70.0,74.0,2.9,62.0,1.5,2.0,6.0,6.0,5.0,6.0,6.5,3.0,1.0,6.0,5.0,6.0,6.0,5.0,6.0,6.0,55.0
75%,21.0,5.0,6.0,85.0,87.0,4.0,91.0,2.2,3.0,8.0,8.0,8.0,8.0,7.3,5.0,1.6,8.0,8.0,8.0,8.0,8.0,8.0,8.0,67.5
max,23.0,8.8,7.0,100.0,100.0,8.9,119.0,5.0,7.0,10.0,10.0,10.0,10.0,10.0,7.0,3.9,10.0,10.0,10.0,10.0,10.0,10.0,10.0,100.0
rel_diff,0.001771,0.001178,0.121945,0.000481,0.006802,0.013656,0.002023,0.031763,0.024371,0.073149,0.077459,0.100035,0.069312,0.001477,0.170504,0.055685,0.080765,0.096706,0.069843,0.07433,0.085937,0.078758,0.077931,0.003259
rdiff_flag,similar,similar,diff,similar,similar,similar,similar,similar,similar,similar,similar,diff,similar,similar,diff,similar,similar,similar,similar,similar,similar,similar,similar,similar


### 2. 기술 통계량 표 저장

In [47]:
desc_df.to_excel("student_performance_qtcheck_desc.xlsx")